# Импорты

In [33]:
import pandas as pd
import numpy as np

# Для формирования отчета в Excel
from openpyxl import Workbook
from openpyxl.styles import PatternFill
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import Alignment
from openpyxl.styles import Font
from openpyxl.styles import Border, Side

id – id проекта \
Причина дубля – причина, почему строки с одним и тем же id встречаются несколько раз \
Колонки с названием месяца – сумма отгрузки проекта в данный месяц \
Account – ФИО ответственного аккаунт-менеджера 

In [34]:
df_financial = pd.read_csv('data/financial_data.csv')
df_financial

,id,Причина дубля,Ноябрь 2022,Декабрь 2022,Январь 2023,Февраль 2023,Март 2023,Апрель 2023,Май 2023,Июнь 2023,Июль 2023,Август 2023,Сентябрь 2023,Октябрь 2023,Ноябрь 2023,Декабрь 2023,Январь 2024,Февраль 2024,Account
0,42,NaN,"36 220,00",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,User_1
1,657,первая часть оплаты,стоп,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,User_1
2,657,вторая часть оплаты,стоп,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,User_1
3,594,NaN,стоп,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,User_1
4,665,NaN,"10 000,00",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,User_1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
446,281,основные работы,"76 550,00","79 950,00","66 900,00","89 150,00","108 450,00","77 100,00","78 800,00","126 740,00","117 730,00","115 860,00","160 770,00","142 490,00","99 125,00","74 350,00","105 775,00","92 065,00",User_2
447,281,доп работы,"21 450,00","13 300,00","15 900,00","19 850,00","17 350,00","14 650,00","15 900,00","3 000,00",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,User_2
448,785,NaN,NaN,NaN,NaN,"5 306,60","12 898,10","5 287,00","10 180,00","8 600,00","3 860,00","8 600,00","700,00","700,00",в ноль,в ноль,NaN,NaN,User_2
449,913,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"58 200,00","58 200,00","58 200,00","58 200,00","58 200,00","58 200,00",NaN,NaN,User_6


id – id проекта \
month – последний месяц реализации проекта \
AM – ФИО ответственного аккаунт-менеджера (данные первичны по отношению к financial_data) 

In [35]:
df_prolongations = pd.read_csv('data/prolongations.csv')
df_prolongations

,id,month,AM
0,42,ноябрь 2022,User_1
1,453,ноябрь 2022,User_1
2,548,ноябрь 2022,User_3
3,87,ноябрь 2022,User_2
4,429,ноябрь 2022,User_2
...,...,...,...
472,955,декабрь 2023,User_6
473,1004,декабрь 2023,User_8
474,281,декабрь 2023,User_2
475,785,декабрь 2023,User_2


# Обработка данных

Получем названия колонок с датами

In [36]:
month_list = df_financial.columns[2:-1].tolist()
month_list

['Ноябрь 2022',
 'Декабрь 2022',
 'Январь 2023',
 'Февраль 2023',
 'Март 2023',
 'Апрель 2023',
 'Май 2023',
 'Июнь 2023',
 'Июль 2023',
 'Август 2023',
 'Сентябрь 2023',
 'Октябрь 2023',
 'Ноябрь 2023',
 'Декабрь 2023',
 'Январь 2024',
 'Февраль 2024']

___________
‘в ноль’ –  отгрузка проекта в данном месяце равна 0, значит для коэффициента 
пролонгации нужно взять отгрузку предыдущего месяца 
(только если все части оплаты равны 0); 

‘стоп’ – проект закончился до истечения срока договора, если у проекта есть “стоп” 
в последний месяц реализации или ранее, то такой проект исключаем из пролонгаций; 

‘end’ – аналогично ‘стоп’


Если в ячейке 'в ноль', нужно взять значение из предыдущего месяца. Такой функционал реализует метод ffill, который работате с пропущенными значениями (у нас NaN). Чтобы заполнить ТОЛЬКО ячейки в которых было 'в ноль', сначала присвоим всем пустым ячейкам другое значение (например большое число). Потом заменим 'в ноль' на NaN и заполним с помощью ffill. Теперь, когда нужные ячейки заполнены, вернем реальные пропуски обратной заменой (большое число на NaN).
_______________

In [37]:
df_financial = df_financial.replace(['стоп', 'end'], 0)
df_financial[month_list] = df_financial[month_list].replace(np.nan, 1000000000000000000000000)
df_financial = df_financial.replace('в ноль', np.nan)
df_financial[month_list] = df_financial[month_list].ffill(axis=1)
df_financial[month_list] = df_financial[month_list].replace(1000000000000000000000000, np.nan)

Теперь преобразуем числа в нормальный формат. Сейчас они хранятся как строки, нам нужны числа. В колонках с числами уберем все пропуски (\xa0 — неразрывный пробел) и заменим запятую на точку, а после преобраузем во float.

In [38]:
for col in month_list:
    df_financial[col] = df_financial[col].astype(str).str.replace(' ','').str.replace(',', '.').str.replace('\xa0','')
    df_financial[col] = df_financial[col].astype(float)

Теперь необходимо сгруппировать по id проекта.

In [39]:
df_financial_group = df_financial.groupby('id').agg({
    **{col: 'sum' for col in month_list},
    'Account': 'first'
}).reset_index()

Мы напишем функцию, которая считает коэффициент по строке. Для расчета коэффициента, нам необходимо знать, в какой месяц заканчивается договор. Для этого объединим данные из двух таблиц (по id). При этом мы сохраним все даты окончания договора для каждого проекта (один проект, может иметь несколько дат).

In [40]:
df_clean = df_prolongations[['id', 'month']].merge(df_financial_group, on='id', how='left')
df_clean.sort_values('id').head(5) # демонстрация того, что у проекта несколько дат

,id,month,Ноябрь 2022,Декабрь 2022,Январь 2023,Февраль 2023,Март 2023,Апрель 2023,Май 2023,Июнь 2023,Июль 2023,Август 2023,Сентябрь 2023,Октябрь 2023,Ноябрь 2023,Декабрь 2023,Январь 2024,Февраль 2024,Account
133,15,февраль 2023,439280.0,439280.0,102433.75,102433.75,102433.75,138158.0,138158.0,102433.75,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,User_5
166,15,март 2023,439280.0,439280.0,102433.75,102433.75,102433.75,138158.0,138158.0,102433.75,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,User_5
24,15,декабрь 2022,439280.0,439280.0,102433.75,102433.75,102433.75,138158.0,138158.0,102433.75,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,User_5
183,15,апрель 2023,439280.0,439280.0,102433.75,102433.75,102433.75,138158.0,138158.0,102433.75,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,User_5
222,15,июнь 2023,439280.0,439280.0,102433.75,102433.75,102433.75,138158.0,138158.0,102433.75,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,User_5


# Расчет коэффициентов

Напишем функции, которые считаю коэффициенты K1 и K2 по строкам, чтобы потом применять эти функции через apply. Также эти функции создают колонки с данными, необходимыми для формирования отчетов по шаблону: к пролонганации (target), пролонгировано (k_revenue).

К1 — отношении суммы отгрузки проектов пролонгированных в первый месяц после завершения к сумме отгрузки последнего месяца реализации всех завершившихся в прошлом месяце проектов.

In [41]:
def K1(row): 
    finish_month = row['month'].strip().capitalize()    # делаем заглавную букву, чтобы найти в списке месяцев / названия столбцов
    finish_index = month_list.index(finish_month)   # определяем индекс ячейки, в которой договор заканчивается
    target = row[month_list[finish_index]]    # в отчете столбец "к пролонгонации" (сумма отгрузки за последний месяц действительного договора)
    if finish_index + 1 >= len(month_list): # если следующий месяц не входит в таблицу, значит это "будущее". мы еще не знаем, продлили договор или нет
        k1 = 0
        k1_revenue = 0

    if target == 0 or pd.isna(target):  # контракт завершен? защита от деления на 0
        k1 = 0
        k1_revenue = 0
    
    else:
        k1 = row[month_list[finish_index+1]] / target # коэффициент K1
        k1_revenue = row[month_list[finish_index+1]]  # в отчете столбец "пролонгировано" (сумма отгрузки в первом месяце, т.к К1)

    return pd.Series([target, k1_revenue, k1])


K2 – отношение суммы отгрузки проектов, пролонгированных во второй месяц к сумме отгрузки последнего месяца проектов, не пролонгированных в первый. 

In [42]:
def K2(row): 
    finish_month = row['month'].strip().capitalize()    # делаем заглавную букву, чтобы найти в списке месяцев / названия столбцов
    k2 = 0
    k2_revenue = 0
    finish_index = month_list.index(finish_month)   # определяем индекс ячейки, в которой договор заканчивается
    target = row[month_list[finish_index]]     # в отчете столбец "к пролонгонации" (сумма отгрузки за последний месяц действительного договора)
    
    if finish_index + 2 >= len(month_list): # если следующий месяц не входит в таблицу, значит это "будущее". мы еще не знаем, продлили договор или нет
        pass

    elif row[month_list[finish_index + 2]] == 0 or row[month_list[finish_index]] == 0:     # защита от деления на 0
        k2 = 0
        k2_revenue = 0

    elif row[month_list[finish_index+1]] == 0 or pd.isna(row[month_list[finish_index + 1]]):    # проверка, что это действительно K2, а не K1 (первый месяц должен быть 0 или NaN)
        k2 = row[month_list[finish_index+2]] / row[month_list[finish_index]]    # коэффициент K2
        k2_revenue = row[month_list[finish_index+2]]   # в отчете столбец "пролонгировано" (сумма отгрузки во втором месяце, т.к К2)

    return pd.Series([target, k2_revenue, k2])


Применяем написанные функции и оставляем только те строки, в которых договор заканчивается в 2023 году (т.к. отчет о пролонгации нужен за 2023 год)

In [43]:
df_clean[['target_k1', 'k1_revenue', 'k1']] = df_clean.apply(K1, axis=1)
df_clean[['target_k2', 'k2_revenue', 'k2']] = df_clean.apply(K2, axis=1)
df_clean = df_clean[df_clean['month'].str.contains('2023')]
df_clean # подготовленная таблица, на основе которой формируем отчеты

,id,month,Ноябрь 2022,Декабрь 2022,Январь 2023,Февраль 2023,Март 2023,Апрель 2023,Май 2023,Июнь 2023,...,Декабрь 2023,Январь 2024,Февраль 2024,Account,target_k1,k1_revenue,k1,target_k2,k2_revenue,k2
91,684,январь 2023,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,User_1,0.0,0.0,0.000000,0.0,0.0,0.000000
92,708,январь 2023,343745.0,501895.0,660475.0,752515.0,795090.0,683985.0,145465.0,163900.0,...,357250.0,0.0,0.0,User_1,660475.0,752515.0,1.139354,660475.0,0.0,0.000000
93,548,январь 2023,674000.0,674000.0,674000.0,674000.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,User_3,674000.0,674000.0,1.000000,674000.0,0.0,0.000000
94,493,январь 2023,47150.0,48850.0,48000.0,0.0,55225.0,55225.0,55225.0,55225.0,...,55225.0,55225.0,55225.0,User_6,48000.0,0.0,0.000000,48000.0,55225.0,1.150521
95,674,январь 2023,106371.0,121069.0,127777.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,User_4,127777.0,0.0,0.000000,127777.0,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
472,955,декабрь 2023,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,17400.0,0.0,0.0,User_6,17400.0,0.0,0.000000,17400.0,0.0,0.000000
473,1004,декабрь 2023,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,User_8,0.0,0.0,0.000000,0.0,0.0,0.000000
474,281,декабрь 2023,98000.0,93250.0,82800.0,109000.0,125800.0,91750.0,94700.0,129740.0,...,74350.0,105775.0,92065.0,User_2,74350.0,105775.0,1.422663,74350.0,0.0,0.000000
475,785,декабрь 2023,0.0,0.0,0.0,5306.6,12898.1,5287.0,10180.0,8600.0,...,700.0,0.0,0.0,User_2,700.0,0.0,0.000000,700.0,0.0,0.000000


# Формирование таблиц и отчетов

Формируем список с месяцами 2023 года, чтобы в отчетах даты шли в правильном порядке

In [44]:
target_month = [item.lower() for item in df_clean.columns if '2023' in item]
target_month

['январь 2023',
 'февраль 2023',
 'март 2023',
 'апрель 2023',
 'май 2023',
 'июнь 2023',
 'июль 2023',
 'август 2023',
 'сентябрь 2023',
 'октябрь 2023',
 'ноябрь 2023',
 'декабрь 2023']

Правильный порядок по пользователям

In [45]:
correct_order = [f'User_{i}' for i in range(1, 11)]
correct_order 

['User_1',
 'User_2',
 'User_3',
 'User_4',
 'User_5',
 'User_6',
 'User_7',
 'User_8',
 'User_9',
 'User_10']

Все отчеты формируются с помощью сводной таблицы (pivot_table). \
"к пролонганации", "пролонгировано" — суммируем за необходимый период (месяц, год) \
После формирования таблиц, сортируем даты по порядку. \
Коэффициенты пересчитываем

### Отчет "Весь отдел"

In [46]:
report_team = df_clean.groupby('month').agg({
    'target_k1': 'sum',
    'k1_revenue': 'sum',
    'target_k2': 'sum',
    'k2_revenue': 'sum'
})

report_team['K1'] = report_team['k1_revenue'] / report_team['target_k1']    # считаем коэффициент для 
report_team['K2'] = report_team['k2_revenue'] / report_team['target_k2']

report_team = report_team.reindex(target_month).reset_index().round(2).fillna(0)
report_team = report_team[['month', 'target_k1', 'k1_revenue', 'K1', 'target_k2', 'k2_revenue', 'K2']]
report_team

,month,target_k1,k1_revenue,K1,target_k2,k2_revenue,K2
0,январь 2023,3329075.50,2539445.00,0.76,3329075.50,122999.08,0.04
1,февраль 2023,1899332.77,1271739.40,0.67,1899332.77,57940.00,0.03
2,март 2023,4157742.20,1512277.90,0.36,4157742.20,0.00,0.00
3,апрель 2023,3657436.50,2082660.75,0.57,3657436.50,38632.50,0.01
4,май 2023,1323850.53,281673.28,0.21,1323850.53,69385.00,0.05
5,июнь 2023,2654147.85,1395166.00,0.53,2654147.85,52315.00,0.02
6,июль 2023,2033915.83,943678.88,0.46,2033915.83,0.00,0.00
7,август 2023,3333288.26,1098704.42,0.33,3333288.26,245379.00,0.07
8,сентябрь 2023,3827610.81,3179423.19,0.83,3827610.81,21400.00,0.01
9,октябрь 2023,1567139.97,816445.47,0.52,1567139.97,114440.00,0.07


In [47]:
columns = pd.MultiIndex.from_tuples([
    ('Месяц', ''),
    ('Пролонгации в первый месяц', 'к пролонгации'),
    ('Пролонгации в первый месяц', 'пролонгировано'),
    ('Пролонгации в первый месяц', 'Коэффициент'),
    ('Пролонгации через месяц', 'к пролонгации'),
    ('Пролонгации через месяц', 'пролонгировано'),
    ('Пролонгации через месяц', 'Коэффициент')
])

# Присваиваем эти колонки нашему отчету
report_team.columns = columns
report_team.iloc[:, 1:7] = report_team.iloc[:, 1:7].round(2).fillna(0)
report_team

Месяц Пролонгации в первый месяц                             \
                               к пролонгации пролонгировано Коэффициент   
0     январь 2023                 3329075.50     2539445.00        0.76   
1    февраль 2023                 1899332.77     1271739.40        0.67   
2       март 2023                 4157742.20     1512277.90        0.36   
3     апрель 2023                 3657436.50     2082660.75        0.57   
4        май 2023                 1323850.53      281673.28        0.21   
5       июнь 2023                 2654147.85     1395166.00        0.53   
6       июль 2023                 2033915.83      943678.88        0.46   
7     август 2023                 3333288.26     1098704.42        0.33   
8   сентябрь 2023                 3827610.81     3179423.19        0.83   
9    октябрь 2023                 1567139.97      816445.47        0.52   
10    ноябрь 2023                 2434794.91     1325978.16        0.54   
11   декабрь 2023                 7707426.85     3571140.19        0.46   

   Пролонгации через месяц                             
             к пролонгации пролонгировано Коэффициент  
0               3329075.50      122999.08        0.04  
1               1899332.77       57940.00        0.03  
2               4157742.20           0.00        0.00  
3               3657436.50       38632.50        0.01  
4               1323850.53       69385.00        0.05  
5               2654147.85       52315.00        0.02  
6               2033915.83           0.00        0.00  
7               3333288.26      245379.00        0.07  
8               3827610.81       21400.00        0.01  
9               1567139.97      114440.00        0.07  
10              2434794.91       86225.50        0.04  
11              7707426.85      456630.00        0.06

### Отчет "Менеджеры за год"

In [48]:
report_manager_year = df_clean.groupby('Account').agg({
    'target_k1': 'sum',
    'k1_revenue': 'sum',
    'target_k2': 'sum',
    'k2_revenue': 'sum',
}).reset_index()

report_manager_year['K1'] = report_manager_year['k1_revenue'] / report_manager_year['target_k1']
report_manager_year['K2'] = report_manager_year['k2_revenue'] / report_manager_year['target_k2']
report_manager_year = report_manager_year[['Account', 'target_k1', 'k1_revenue', 'K1', 'target_k2', 'k2_revenue', 'K2']]

report_manager_year = report_manager_year.set_index('Account').reindex(correct_order).reset_index()
report_manager_year = report_manager_year.sort_values(['K1', 'K2'], ascending=[False, False])
report_manager_year

,Account,target_k1,k1_revenue,K1,target_k2,k2_revenue,K2
8,User_9,98492.00,109442.52,1.111182,98492.00,0.00,0.000000
6,User_7,2268220.21,1410674.83,0.621930,2268220.21,67774.08,0.029880
1,User_2,7582390.99,4707502.42,0.620847,7582390.99,0.00,0.000000
5,User_6,4424212.39,2728241.30,0.616661,4424212.39,412350.50,0.093203
2,User_3,2566389.68,1468179.75,0.572080,2566389.68,136730.00,0.053277
9,User_10,128250.00,73050.00,0.569591,128250.00,0.00,0.000000
0,User_1,12347860.86,5861909.79,0.474731,12347860.86,366310.00,0.029666
3,User_4,6560476.90,3046753.40,0.464410,6560476.90,243549.00,0.037124
4,User_5,1949468.95,612578.63,0.314228,1949468.95,38632.50,0.019817
7,User_8,0.00,0.00,NaN,0.00,0.00,NaN


In [49]:
# Определяем структуру шапки
columns = pd.MultiIndex.from_tuples([
    ('Менеджер', ''),
    ('Пролонгации в первый месяц', 'к пролонгации'),
    ('Пролонгации в первый месяц', 'пролонгировано'),
    ('Пролонгации в первый месяц', 'Коэффициент'),
    ('Пролонгации через месяц', 'к пролонгации'),
    ('Пролонгации через месяц', 'пролонгировано'),
    ('Пролонгации через месяц', 'Коэффициент')
])

# Присваиваем эти колонки нашему отчету
report_manager_year.columns = columns

report_manager_year.iloc[:, 1:7] = report_manager_year.iloc[:, 1:7].round(2).fillna(0)
report_manager_year


Менеджер Пролонгации в первый месяц                             \
                        к пролонгации пролонгировано Коэффициент   
8   User_9                   98492.00      109442.52        1.11   
6   User_7                 2268220.21     1410674.83        0.62   
1   User_2                 7582390.99     4707502.42        0.62   
5   User_6                 4424212.39     2728241.30        0.62   
2   User_3                 2566389.68     1468179.75        0.57   
9  User_10                  128250.00       73050.00        0.57   
0   User_1                12347860.86     5861909.79        0.47   
3   User_4                 6560476.90     3046753.40        0.46   
4   User_5                 1949468.95      612578.63        0.31   
7   User_8                       0.00           0.00        0.00   

  Пролонгации через месяц                             
            к пролонгации пролонгировано Коэффициент  
8                98492.00           0.00        0.00  
6              2268220.21       67774.08        0.03  
1              7582390.99           0.00        0.00  
5              4424212.39      412350.50        0.09  
2              2566389.68      136730.00        0.05  
9               128250.00           0.00        0.00  
0             12347860.86      366310.00        0.03  
3              6560476.90      243549.00        0.04  
4              1949468.95       38632.50        0.02  
7                    0.00           0.00        0.00

### Отчет "Менеджеры по месяцам"

In [50]:
manager_month_k1_temp = df_clean.groupby(['Account', 'month']).agg({
    'target_k1': 'sum',
    'k1_revenue': 'sum'
}).reset_index()

manager_month_k1_temp['K1'] = manager_month_k1_temp['k1_revenue'] / manager_month_k1_temp['target_k1']

manager_month_k1 = manager_month_k1_temp.pivot(
    index='Account', 
    columns='month', 
    values='K1')

manager_month_k1 = manager_month_k1.reindex(columns=target_month).fillna(0).reset_index().round(2)
manager_month_k1 = manager_month_k1.set_index('Account').reindex(correct_order).reset_index()

manager_month_k1

month,Account,январь 2023,февраль 2023,март 2023,апрель 2023,май 2023,июнь 2023,июль 2023,август 2023,сентябрь 2023,октябрь 2023,ноябрь 2023,декабрь 2023
0,User_1,1.04,0.50,0.34,0.46,0.00,0.46,0.43,0.13,0.88,0.62,0.32,0.54
1,User_2,0.38,1.00,1.07,0.54,0.77,0.62,0.53,0.40,0.86,0.18,0.79,0.41
2,User_3,1.00,0.00,0.37,1.33,0.00,1.27,0.00,0.00,0.26,0.00,0.00,0.00
3,User_4,0.80,0.60,0.04,0.59,0.00,0.09,0.31,0.46,0.84,0.00,0.60,0.42
4,User_5,0.00,0.45,0.32,0.78,0.00,0.00,0.38,1.00,0.00,0.00,0.00,0.00
5,User_6,0.00,0.99,0.97,0.70,0.44,0.00,0.48,0.77,1.06,0.84,0.36,0.41
6,User_7,0.30,1.34,0.58,1.09,0.86,1.14,0.00,1.16,1.31,0.49,0.27,0.59
7,User_8,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
8,User_9,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.11,0.00
9,User_10,0.00,0.00,0.00,0.00,0.00,0.00,0.94,0.00,0.00,0.00,0.00,0.00


In [51]:
manager_month_k2_temp = df_clean.groupby(['Account', 'month']).agg({
    'target_k2': 'sum',
    'k2_revenue': 'sum'
}).reset_index()

manager_month_k2_temp['K2'] = manager_month_k2_temp['k2_revenue'] / manager_month_k2_temp['target_k2']

manager_month_k2 = manager_month_k2_temp.pivot(
    index='Account', 
    columns='month', 
    values='K2')

manager_month_k2 = manager_month_k2.reindex(columns=target_month).fillna(0).reset_index().round(2)
manager_month_k2 = manager_month_k2.set_index('Account').reindex(correct_order).reset_index()
manager_month_k2

month,Account,январь 2023,февраль 2023,март 2023,апрель 2023,май 2023,июнь 2023,июль 2023,август 2023,сентябрь 2023,октябрь 2023,ноябрь 2023,декабрь 2023
0,User_1,0.00,0.05,0.0,0.00,0.00,0.00,0.0,0.06,0.00,0.15,0.00,0.05
1,User_2,0.00,0.00,0.0,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00
2,User_3,0.00,0.00,0.0,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,1.03
3,User_4,0.00,0.05,0.0,0.00,0.00,0.00,0.0,0.32,0.02,0.00,0.36,0.00
4,User_5,0.00,0.00,0.0,0.22,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00
5,User_6,1.15,0.00,0.0,0.00,0.29,1.55,0.0,0.00,0.00,0.10,0.08,0.12
6,User_7,0.36,0.00,0.0,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00
7,User_8,0.00,0.00,0.0,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00
8,User_9,0.00,0.00,0.0,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00
9,User_10,0.00,0.00,0.0,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00


In [52]:
columns_all = pd.MultiIndex.from_tuples([
    ('Менеджер', ''),
    ('Месяц', 'Январь'),
    ('Месяц', 'Февраль'),
    ('Месяц', 'Март'),
    ('Месяц', 'Апрель'),
    ('Месяц', 'Май'),
    ('Месяц', 'Июнь'),
    ('Месяц', 'Июль'),
    ('Месяц', 'Август'),
    ('Месяц', 'Сентябрь'),
    ('Месяц', 'Октябрь'),
    ('Месяц', 'Ноябрь'),
    ('Месяц', 'Декабрь')
])

# Присваиваем эти колонки нашему отчету
manager_month_k1.columns = columns_all
manager_month_k2.columns = columns_all
manager_month_k1


Менеджер  Месяц                                                         \
           Январь Февраль  Март Апрель   Май  Июнь  Июль Август Сентябрь   
0   User_1   1.04    0.50  0.34   0.46  0.00  0.46  0.43   0.13     0.88   
1   User_2   0.38    1.00  1.07   0.54  0.77  0.62  0.53   0.40     0.86   
2   User_3   1.00    0.00  0.37   1.33  0.00  1.27  0.00   0.00     0.26   
3   User_4   0.80    0.60  0.04   0.59  0.00  0.09  0.31   0.46     0.84   
4   User_5   0.00    0.45  0.32   0.78  0.00  0.00  0.38   1.00     0.00   
5   User_6   0.00    0.99  0.97   0.70  0.44  0.00  0.48   0.77     1.06   
6   User_7   0.30    1.34  0.58   1.09  0.86  1.14  0.00   1.16     1.31   
7   User_8   0.00    0.00  0.00   0.00  0.00  0.00  0.00   0.00     0.00   
8   User_9   0.00    0.00  0.00   0.00  0.00  0.00  0.00   0.00     0.00   
9  User_10   0.00    0.00  0.00   0.00  0.00  0.00  0.94   0.00     0.00   

                          
  Октябрь Ноябрь Декабрь  
0    0.62   0.32    0.54  
1    0.18   0.79    0.41  
2    0.00   0.00    0.00  
3    0.00   0.60    0.42  
4    0.00   0.00    0.00  
5    0.84   0.36    0.41  
6    0.49   0.27    0.59  
7    0.00   0.00    0.00  
8    0.00   1.11    0.00  
9    0.00   0.00    0.00

In [53]:
manager_month_k2

Менеджер  Месяц                                                       \
           Январь Февраль Март Апрель   Май  Июнь Июль Август Сентябрь   
0   User_1   0.00    0.05  0.0   0.00  0.00  0.00  0.0   0.06     0.00   
1   User_2   0.00    0.00  0.0   0.00  0.00  0.00  0.0   0.00     0.00   
2   User_3   0.00    0.00  0.0   0.00  0.00  0.00  0.0   0.00     0.00   
3   User_4   0.00    0.05  0.0   0.00  0.00  0.00  0.0   0.32     0.02   
4   User_5   0.00    0.00  0.0   0.22  0.00  0.00  0.0   0.00     0.00   
5   User_6   1.15    0.00  0.0   0.00  0.29  1.55  0.0   0.00     0.00   
6   User_7   0.36    0.00  0.0   0.00  0.00  0.00  0.0   0.00     0.00   
7   User_8   0.00    0.00  0.0   0.00  0.00  0.00  0.0   0.00     0.00   
8   User_9   0.00    0.00  0.0   0.00  0.00  0.00  0.0   0.00     0.00   
9  User_10   0.00    0.00  0.0   0.00  0.00  0.00  0.0   0.00     0.00   

                          
  Октябрь Ноябрь Декабрь  
0    0.15   0.00    0.05  
1    0.00   0.00    0.00  
2    0.00   0.00    1.03  
3    0.00   0.36    0.00  
4    0.00   0.00    0.00  
5    0.10   0.08    0.12  
6    0.00   0.00    0.00  
7    0.00   0.00    0.00  
8    0.00   0.00    0.00  
9    0.00   0.00    0.00

# Оформление отчетов в Excel

In [54]:
# Цвет для заголовков
yellow_fill = PatternFill(start_color='FFF2CC',
                          end_color='FFF2CC',
                          fill_type='solid')

# Цвет для заголовков (коэффициенты)            
yellow_dark = PatternFill(start_color='FFD966',
                          end_color='FFD966',
                          fill_type='solid')

In [55]:
# Жирный шрифт для заголовков
bold_font = Font(bold=True)

In [56]:
# Рамка для заголовков
thin_side = Side(border_style="thin", color="000000")
full_border = Border(top=thin_side, left=thin_side, right=thin_side, bottom=thin_side)

Создаем книгу в Excel и листы для отчетов

In [57]:
wb = Workbook()
ws1 = wb.active
ws1.title = 'Весь отдел'
ws2 = wb.create_sheet(title='Менеджеры за год')
ws3 = wb.create_sheet(title='Менеджеры по месяцам')

### Оформление отчета "Весь отдел"

In [58]:
for r in dataframe_to_rows(report_team, index=False, header=True):  # добавляем строки из таблицы на лист 1
    ws1.append(r)
    
current_row = ws1.min_row   # берем самую верхнюю строку
for col in range(1, 8):     # красим ячейки и настраиваем жирность шрифта
    ws1.cell(row=current_row, column=col).fill = yellow_fill
    ws1.cell(row=current_row+1, column=col).fill = yellow_fill
    ws1.cell(row=current_row, column=col).font = bold_font
    ws1.cell(row=current_row+1, column=col).font = bold_font

ws1.merge_cells('A1:A2')    # объединяем ячейки
ws1['A1'].alignment = Alignment(horizontal='center', vertical='center')     # выравниваем надпись в ячейке

ws1.merge_cells('B1:D1')     # объединяем ячейки
ws1['B1'].alignment = Alignment(horizontal='center', vertical='center')     # выравниваем надпись в ячейке

ws1.merge_cells('E1:G1')     # объединяем ячейки
ws1['E1'].alignment = Alignment(horizontal='center', vertical='center')     # выравниваем надпись в ячейке

for col in range(1, 8):     # добавляем рамку
    cell = ws1.cell(row=current_row, column=col)
    cell.border = full_border
    cell = ws1.cell(row=current_row+1, column=col)
    cell.border = full_border

### Оформление отчета "Менеджеры за год"

In [59]:
for r in dataframe_to_rows(report_manager_year, index=False, header=True):   # добавляем строки из таблицы на лист 2
    ws2.append(r)
    
current_row = ws2.min_row   # берем самую верхнюю строку
for col in range(1, 8):     # красим ячейки и настраиваем жирность шрифта
    ws2.cell(row=current_row, column=col).fill = yellow_fill
    ws2.cell(row=current_row+1, column=col).fill = yellow_fill
    ws2.cell(row=current_row, column=col).font = bold_font
    ws2.cell(row=current_row+1, column=col).font = bold_font

ws2.merge_cells('A1:A2')     # объединяем ячейки
ws2['A1'].alignment = Alignment(horizontal='center', vertical='center')     # выравниваем надпись в ячейке

ws2.merge_cells('B1:D1')     # объединяем ячейки
ws2['B1'].alignment = Alignment(horizontal='center', vertical='center')     # выравниваем надпись в ячейке

ws2.merge_cells('E1:G1')     # объединяем ячейки
ws2['E1'].alignment = Alignment(horizontal='center', vertical='center')     # выравниваем надпись в ячейке

for col in range(1, 8):     # добавляем рамку
    cell = ws2.cell(row=current_row, column=col)
    cell.border = full_border
    cell = ws2.cell(row=current_row+1, column=col)
    cell.border = full_border

### Оформление отчета "Менеджеры по месяцам"

In [61]:
wb.save("data/report.xlsx")  # сохраняем результат в книгу Excel